# Séance 4 - Bonus : Réseau récurrent avec Embedding

Dans cette séance nous avons entraîné un modèle à copier le style de poésie de Beaudelaire, spécifiquement l'oeuvre *Les fleurs du mal*. On souhaite voir ici comment utiliser la couche [`Embedding`](https://keras.io/api/layers/core_layers/embedding/) et ce que l'on peut faire avec.

Commençons par importer les données.

In [6]:
import keras
import numpy as np
import seaborn as sns

sns.set(style="whitegrid")


start = False
book = open("Beaudelaire.txt", encoding="utf8")  # noqa: SIM115
lines = book.readlines()
verses = []

for line in lines:
    line_striped = line.strip().lower()
    if "AU LECTEUR".lower() in line_striped and not start:
        start = True
    if (
        "End of the Project Gutenberg EBook of Les Fleurs du Mal, by Charles Baudelaire".lower()
        in line_striped
    ):
        break
    if not start or len(line_striped) == 0:
        continue
    verses.append(line_striped)

book.close()
text = " ".join(verses)
characters = sorted(set(text))
n_characters = len(characters)

Dans le TP principal nous avons one-hot encodé le texte. La couche [`Embedding`](https://keras.io/api/layers/core_layers/embedding/) prend en entrée une séquence d'entier. Ainsi, nous devons changer la manière de construire $X$ et $y$.

**Consigne** : En s'inspirant du travail précédent, construire la matrice d'informations $X$ et le vecteur réponse $y$. Puis on scindera le dataset en un dataset d'entraînement et de validation.

In [ ]:
# Create character to index and index to character mappings
char_to_idx = {char: idx for idx, char in enumerate(characters)}
idx_to_char = dict(enumerate(characters))

# Parameters
sequence_length = 40

# Create sequences
X = []
y = []

for i in range(len(text) - sequence_length):
    # Input sequence: convert characters to indices
    sequence = text[i : i + sequence_length]
    X.append([char_to_idx[char] for char in sequence])

    # Target: next character as index
    target = text[i + sequence_length]
    y.append(char_to_idx[target])

X = np.array(X)
y = np.array(y)

# Split into training and validation sets
split_ratio = 0.8
split_index = int(len(X) * split_ratio)

X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: (108720, 40)
y_train shape: (108720,)
X_val shape: (27181, 40)
y_val shape: (27181,)


La couche [`Embedding`](https://keras.io/api/layers/core_layers/embedding/) a comme paramètre :
* *input_dim* : la taille du vocabulaire que l'on considère, ici *n_characters*
* *output_dim* : la dimension de l'embedding, autrement dit chaque caractère sera représenté comme un vecteur de *output_dim* dimension

On souhaite mesurer l'impact du paramètre *output_dim*. 

**Consigne** : Définir une fonction `get_model` qui prend en paramètre:
* *dimension* : un entier qui correspond à la dimension de sortie de l'embedding
* *vocabulary_size* : la taille du vocabulaire

La fonction renvoie un réseau de neurones récurrents avec une couche d'embedding paramétré en accord avec les paramètres de la fonction. On essayera de faire un modèle de taille raisonnable.


In [9]:
def get_model(dimension: int, vocabulary_size: int) -> keras.Model:
    """Create and return a SimpleRNN Keras model.

    Args:
        dimension (int): The embedding dimension.
        vocabulary_size (int): The size of the vocabulary.

    Returns:
        keras.Model: The constructed Keras model.

    """
    model = keras.Sequential()
    model.add(
        keras.layers.Embedding(
            input_dim=vocabulary_size,
            output_dim=dimension,
        )
    )
    model.add(keras.layers.SimpleRNN(128, return_sequences=False))
    model.add(keras.layers.Dense(vocabulary_size, activation="softmax"))
    return model

**Consigne** : Écrire une boucle d'entraînement qui va stocker dans une liste le maximum atteint lors de l'entraînement jusqu'à 10 époques. Chaque élément de la liste correspondra à un dictionnaire avec pour clé:
* *dimension*: la dimension de l'embedding
* *val_loss*: la valeur de loss minimale atteinte sur le dataset de validation au cours de l'entraînement

In [10]:
dimension = 64
vocabulary_size = n_characters

model = get_model(dimension, vocabulary_size)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Consigne** : Modifier la structure de results pour correspondre à une liste de tuple où on a la moyenne et l'écart-type pour chaque entraînement pour une dimension précise.

**Consigne** : Visualiser puis commenter les résultats.